In [6]:
import scanpy as sc

hlca_data_path = "scAnnotation_Datasets/Datasets/HLCA/hlca_core.h5ad"

hlca_data = sc.read_h5ad(hlca_data_path)

In [7]:
print(hlca_data)

AnnData object with n_obs × n_vars = 584944 × 27402
    obs: 'ann_level_1', 'ann_level_2', 'ann_level_3', 'ann_level_4', 'ann_level_5', 'ann_finest_level', 'cell_type', 'scanvi_label', 'donor_id', 'sample', 'dataset', 'study', 'subject_type', 'lung_condition', 'sex', 'age_or_mean_of_age_range', 'BMI', 'smoking_status', 'self_reported_ethnicity', 'tissue', 'tissue_level_2', 'tissue_sampling_method', 'assay', 'fresh_or_frozen', 'log10_total_counts', 'n_genes_detected', 'size_factors'
    var: 'ensembl_id', 'feature_name', 'feature_type'
    uns: 'batch_condition', 'citation', 'default_embedding', 'is_pre_analysis', 'organism', 'organism_ontology_term_id', 'schema_reference', 'schema_version', 'title'


In [8]:
print(hlca_data.var.head())

             ensembl_id feature_name    feature_type
TSPAN6  ENSG00000000003       TSPAN6  protein_coding
TNMD    ENSG00000000005         TNMD  protein_coding
DPM1    ENSG00000000419         DPM1  protein_coding
SCYL3   ENSG00000000457        SCYL3  protein_coding
FIRRM   ENSG00000000460        FIRRM  protein_coding


In [9]:
print(hlca_data.obs.head())

                      ann_level_1          ann_level_2  \
GCGACCATCCCTAACC_SC22      Immune              Myeloid   
P2_1_GCGCAACCAGTTAACC      Immune             Lymphoid   
GCTCTGTAGTGCTGCC_SC27  Epithelial  Alveolar epithelium   
P2_8_TTAGGACGTTCAGGCC      Immune              Myeloid   
CTTGATTGTCAGTTTG_T164  Epithelial    Airway epithelium   

                                   ann_level_3           ann_level_4  \
GCGACCATCCCTAACC_SC22              Macrophages  Alveolar macrophages   
P2_1_GCGCAACCAGTTAACC  Innate lymphoid cell NK              NK cells   
GCTCTGTAGTGCTGCC_SC27                      AT2                  None   
P2_8_TTAGGACGTTCAGGCC              Macrophages  Alveolar macrophages   
CTTGATTGTCAGTTTG_T164                    Basal            Suprabasal   

                              ann_level_5      ann_finest_level  \
GCGACCATCCCTAACC_SC22                None  Alveolar macrophages   
P2_1_GCGCAACCAGTTAACC                None              NK cells   
GCTCTGTAGTGCTGCC_

In [ ]:
import scanpy as sc

pbmc_data_path = "scAnnotation_Datasets/Datasets/PBMC_CiteSeqRef/pbmc_citeseq_ref_2021.h5ad"
pbmc_data = sc.read_h5ad(pbmc_data_path)
print(pbmc_data)

print(pbmc_data.var.head())

AnnData object with n_obs × n_vars = 161764 × 33538
    obs: 'celltype.l1', 'celltype.l2', 'celltype.l3', 'donor', 'time', 'lane', 'Phase', 'nCount_RNA', 'nFeature_RNA', 'nCount_ADT', 'nFeature_ADT'
    var: 'gene_symbol', 'feature_type'
    uns: 'ADT_names', 'dataset', 'source_geo'
    obsm: 'protein_counts'
             gene_symbol     feature_type
MIR1302-2HG  MIR1302-2HG  Gene Expression
FAM138A          FAM138A  Gene Expression
OR4F5              OR4F5  Gene Expression
AL627309.1    AL627309.1  Gene Expression
AL627309.3    AL627309.3  Gene Expression


In [ ]:
print(hlca_data.obs.head())

In [3]:
import scanpy as sc

cell_idx = 13748
pbmc_data_path = "Datasets/PBMC_CiteSeqRef/pbmc_citeseq_ref_2021.h5ad"

pbmc_data = sc.read_h5ad(pbmc_data_path)
data = pbmc_data[cell_idx]
print(data)

View of AnnData object with n_obs × n_vars = 1 × 33538
    obs: 'celltype.l1', 'celltype.l2', 'celltype.l3', 'donor', 'time', 'lane', 'Phase', 'nCount_RNA', 'nFeature_RNA', 'nCount_ADT', 'nFeature_ADT'
    var: 'gene_symbol', 'feature_type'
    uns: 'ADT_names', 'dataset', 'source_geo'
    obsm: 'protein_counts'


In [4]:
import scanpy as sc

from sc_annotation.config import ExperimentConfig
from sc_annotation.pipeline import ensure_global_metrics, _resolve_cell_indices, _select_genes, _build_prompt
from sc_annotation.filtering import build_gene_mask
from sc_annotation.selection import MarkerPanel
from sc_annotation.smoothing import build_knn_index, compute_pseudobulk
from sc_annotation.adt import build_adt_adata, normalize_adt, get_top_proteins as get_top_proteins_adt

# 1) Load the same experiment config used by pipeline
config_path = "configs/pbmc_l2_knn.yaml"
config = ExperimentConfig.from_yaml(config_path)

# You can override index here
cell_idx = 13748

# 2) Load dataset from config (same as pipeline)
adata = sc.read_h5ad(config.dataset_path)

# 3) Pipeline preprocessing
ensure_global_metrics(adata, config.global_metrics)
gene_mask = build_gene_mask(adata, config.filter)

marker_panel = None
if config.selection.marker_panel_path:
    marker_panel = MarkerPanel.from_yaml(config.selection.marker_panel_path)

adt_adata = None
if "adt" in config.selection.modalities:
    try:
        adt_adata = build_adt_adata(adata)
        normalize_adt(adt_adata, method="clr", inplace=True)
    except KeyError:
        adt_adata = None

neighbor_matrix = None
label_to_indices = None
if config.input.mode == "knn_smoothed":
    neighbor_matrix = build_knn_index(
        adata,
        k=config.input.knn_k,
        use_rep=config.input.knn_use_rep,
        inplace=True,
    )
elif config.input.mode == "pseudobulk":
    label_to_indices = compute_pseudobulk(adata, config.evaluation.label_col)

# 4) Build per-cell prompt exactly like stage-1 in pipeline (no inference)
cell_indices = _resolve_cell_indices(
    adata,
    cell_idx,
    config.input,
    neighbor_matrix,
    label_to_indices,
    config.evaluation.label_col,
)

gene_sel = _select_genes(
    adata,
    cell_indices,
    config.selection,
    gene_mask,
    marker_panel,
)

proteins = []
if adt_adata is not None:
    proteins = get_top_proteins_adt(adt_adata, cell_indices, n_top=20)

# Keep this list empty to match default free-form pipeline behavior
cell_type_list = []
stage1_prompt = _build_prompt(gene_sel, config.selection, proteins, config.tissue, cell_type_list)

print(f"cell_idx={cell_idx}, true_label={adata.obs[config.evaluation.label_col].iloc[cell_idx]}")
print("\n===== STAGE-1 PROMPT (NO INFERENCE) =====\n")
print(stage1_prompt)

Annotating genes …
Computing global metrics (first time only) …
INFO: Computing lognorm layer...
INFO: Computing population statistics...
INFO: Computing highly variable genes...
INFO: Computing Gini index...
  Done in 33.7s  |  var cols: ['gene_symbol', 'feature_type', 'is_mt', 'is_ribo', 'is_hb', 'is_tcr_vdj', 'is_sex_chr', 'mean_expr', 'std_expr', 'pct_cells', 'idf', 'is_low_expr', 'is_hvg', 'gini']
INFO: Computing PCA embedding with 50 components...
INFO: Building kNN index with k=10 on embedding 'X_pca'...
cell_idx=13748, true_label=CD16 Mono

===== STAGE-1 PROMPT (NO INFERENCE) =====

Tissue context: PBMC

Top expressed genes (ranked by expression level):
    1. MALAT1 (5.910)
    2. FTL (5.449)
    3. TMSB4X (5.005)
    4. B2M (4.951)
    5. FTH1 (4.853)
    6. TMSB10 (4.308)
    7. ACTB (4.169)
    8. S100A4 (4.119)
    9. EEF1A1 (4.064)
   10. S100A6 (3.967)
   11. CST3 (3.890)
   12. AIF1 (3.799)
   13. TPT1 (3.720)
   14. LST1 (3.644)
   15. NEAT1 (3.577)
   16. CTSS (3.565)

In [5]:
import numpy as np
import pandas as pd
from scipy import sparse

# Assume `adata` and `cell_idx` are already defined in the previous cell.
# If not, uncomment the next two lines:
# import scanpy as sc
# adata = sc.read_h5ad("Datasets/PBMC_CiteSeqRef/pbmc_citeseq_ref_2021.h5ad")

def _to_dense_1d(x):
    if sparse.issparse(x):
        return x.toarray().ravel()
    return np.asarray(x).ravel()

cell = adata[cell_idx]

print("===== Cell Basic Info =====")
print(f"cell_idx: {cell_idx}")
print(f"barcode: {adata.obs_names[cell_idx]}")
print(f"shape: {cell.shape}")

print("\n===== Cell obs (all metadata fields) =====")
obs_row = adata.obs.iloc[cell_idx]
obs_df = obs_row.to_frame(name="value")
print(obs_df)

# Expression vector and gene symbols
x = _to_dense_1d(cell.X)
if "gene_symbol" in adata.var.columns:
    gene_names = adata.var["gene_symbol"].astype(str).values
else:
    gene_names = adata.var_names.astype(str).values

expr_df = pd.DataFrame({"gene": gene_names, "expr": x})
expr_df = expr_df.sort_values("expr", ascending=False)

print("\n===== Top 30 expressed genes (current matrix) =====")
print(expr_df.head(30).to_string(index=False))

# Non-zero genes
nz_df = expr_df[expr_df["expr"] > 0]
print(f"\nNon-zero genes: {len(nz_df)}")
print("\n===== Top 30 non-zero genes =====")
print(nz_df.head(30).to_string(index=False))

# Optional: raw matrix if available
if adata.raw is not None:
    raw_cell = adata.raw[cell_idx]
    raw_x = _to_dense_1d(raw_cell.X)
    raw_genes = np.asarray(adata.raw.var_names).astype(str)
    raw_df = pd.DataFrame({"gene": raw_genes, "expr_raw": raw_x}).sort_values("expr_raw", ascending=False)

    print("\n===== Top 30 genes in adata.raw =====")
    print(raw_df.head(30).to_string(index=False))
else:
    print("\nadata.raw is None, skip raw expression view.")

===== Cell Basic Info =====
cell_idx: 13748
barcode: L1_TTTGTTGCAGCGAGTA
shape: (1, 33538)

===== Cell obs (all metadata fields) =====
                  value
celltype.l1        Mono
celltype.l2   CD16 Mono
celltype.l3   CD16 Mono
donor                P4
time                  3
lane                 L1
Phase                 S
nCount_RNA      10643.0
nFeature_RNA       2967
nCount_ADT      10637.0
nFeature_ADT        216

===== Top 30 expressed genes (current matrix) =====
   gene  expr
    FTL   322
 MALAT1   275
 TMSB4X   218
    B2M   141
   ACTB   113
 S100A4   110
   FTH1   109
 TMSB10   109
  RPS19   104
 MT-CO1    92
 MT-CO2    90
 S100A6    84
MT-ATP6    80
  RPLP1    71
   LST1    70
   CST3    65
  RPL28    59
  RPL41    58
 MT-CO3    55
  RPL39    52
  RPS12    51
  RPS27    48
 MT-ND3    48
  RPS18    44
  RPS29    42
   OAZ1    42
   AIF1    41
   CTSS    41
 EEF1A1    41
  RPL21    40

Non-zero genes: 2967

===== Top 30 non-zero genes =====
   gene  expr
    FTL   322
 MALA